In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
# --- 1) Daten laden ---

drl_data = pd.read_csv(
    "data/data_2019-01-01_2024-01-01_hourly.csv",
    parse_dates=["time"],
    index_col="time",
)
drl_data.index = pd.to_datetime(drl_data.index, utc=True)

In [ ]:
# Optimiert für Zeitreihenanalyse
exaa_data_15min = pd.read_csv(
    "data/exaa_qh_prices/exaa_15min_prices_2019_2025_utc.csv",
    parse_dates=['datetime_utc'],
    index_col='datetime_utc'
)
exaa_data_15min.index = pd.to_datetime(exaa_data_15min.index, utc=True)

exaa_data = exaa_data_15min.resample('H').mean()

exaa_data = exaa_data.rename(columns={'price_eur_mwh': 'exaa_15min_de_lu_eur_per_mwh'})

drl_data = drl_data.join(exaa_data, how='left')

In [ ]:
ri_profit1 = pd.read_csv("output/single_market/rolling_intrinsic/ri_basic/qh/2019/bs15cr1rto0.86mc365mt10/profit.csv")
ri_profit2 = pd.read_csv("output/single_market/rolling_intrinsic/ri_basic/qh/2023/bs15cr1rto0.86mc365mt10/profit.csv")

ri_profit = pd.concat([ri_profit1, ri_profit2], ignore_index=True)
ri_profit = ri_profit.set_index("day")  # aktuell: dtype=object mit +01:00 im String


In [ ]:
# 1) Index-Strings auf 'YYYY-MM-DD HH:MM:SS' kürzen (Offset abschneiden)
idx = ri_profit.index.astype(str).str.slice(0, 19)
# z.B. '2019-01-01 00:00:00+01:00' -> '2019-01-01 00:00:00'

# 2) In echte Datumswerte umwandeln (noch ohne Zeitzone)
idx = pd.to_datetime(idx)  # dtype: datetime64[ns], immer 00:00:00

# 3) Zeitzone als UTC hinzufügen
idx = idx.tz_localize("UTC")  # dtype: datetime64[ns, UTC]

# 4) Zurück als Index setzen
ri_profit.index = idx

# Optional: Index-Name an den anderen DF anpassen
ri_profit.index.name = "date"

In [ ]:
ri_profit.index

DatetimeIndex(['2019-01-01 00:00:00+00:00', '2019-01-02 00:00:00+00:00',
               '2019-01-03 00:00:00+00:00', '2019-01-04 00:00:00+00:00',
               '2019-01-05 00:00:00+00:00', '2019-01-06 00:00:00+00:00',
               '2019-01-07 00:00:00+00:00', '2019-01-08 00:00:00+00:00',
               '2019-01-09 00:00:00+00:00', '2019-01-10 00:00:00+00:00',
               ...
               '2023-12-22 00:00:00+00:00', '2023-12-23 00:00:00+00:00',
               '2023-12-24 00:00:00+00:00', '2023-12-25 00:00:00+00:00',
               '2023-12-26 00:00:00+00:00', '2023-12-27 00:00:00+00:00',
               '2023-12-28 00:00:00+00:00', '2023-12-29 00:00:00+00:00',
               '2023-12-30 00:00:00+00:00', '2023-12-31 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='date', length=1826, freq=None)

In [ ]:
drl_data.drop(columns=["exaa_15min_de_lu_eur_per_mwh", "id_full_qh"], inplace=True)

In [ ]:
# --- 2) Residual Load berechnen ---

drl_data["residual_load"] = (
    drl_data["load_forecast_d_minus_1_1000_total_de_lu_mw"]
    - drl_data["pv_forecast_d_minus_1_1000_de_lu_mw"]
    - drl_data["wind_offshore_forecast_d_minus_1_1000_de_lu_mw"]
    - drl_data["wind_onshore_forecast_d_minus_1_1000_de_lu_mw"]
)




In [ ]:
# --- 3) Daily Aggregation über Resample ---

agg_features = [
    "epex_spot_60min_de_lu_eur_per_mwh",
    "exaa_15min_de_lu_eur_per_mwh",
    "load_forecast_d_minus_1_1000_total_de_lu_mw",
    "pv_forecast_d_minus_1_1000_de_lu_mw",
    "wind_offshore_forecast_d_minus_1_1000_de_lu_mw",
    "wind_onshore_forecast_d_minus_1_1000_de_lu_mw",
    "residual_load",
    "id_full_h",
   # "id_full_qh",
]

agg_dict = {
    col: [
        "mean",
        "std",
        "min",
        "max",
        ("q25", lambda x: x.quantile(0.25)),
        ("q50", lambda x: x.quantile(0.50)),
        ("q75", lambda x: x.quantile(0.75)),
    ]
    for col in agg_features
}

daily_agg = (
    drl_data
    .resample("D")      # täglich auf Index 'time'
    .agg(agg_dict)
)

daily_agg.columns = [
    f"{col}_{stat}" for col, stat in daily_agg.columns.to_flat_index()
]
daily_agg.index.name = "date"

daily_agg["epex_spread"] = (
    daily_agg["epex_spot_60min_de_lu_eur_per_mwh_max"]
    - daily_agg["epex_spot_60min_de_lu_eur_per_mwh_min"]
)

daily_agg["residual_load_spread"] = (
    daily_agg["residual_load_max"] - daily_agg["residual_load_min"]
)

daily_agg["id_full_h_spread"] = (
    daily_agg["id_full_h_max"] - daily_agg["id_full_h_min"]
)

daily_agg["day_of_week"] = daily_agg.index.dayofweek
daily_agg["month"] = daily_agg.index.month
daily_agg["year"] = daily_agg.index.year


In [ ]:
def add_daily_delta_features(
    df,
    column,
    prefix=None,
):
    """
    Berechnet für eine Zeitreihe:
    - tägliche Summe der absoluten Deltas
    - maximale Intraday-Rampe
    """
    if prefix is None:
        prefix = column

    delta = (
        df[column]
        .groupby(df.index.date)
        .diff()
        .abs()
    )

    daily = pd.DataFrame({
        f"{prefix}_delta_sum": delta.resample("D").sum(),
        f"{prefix}_delta_max": delta.resample("D").max(),
    })

    return daily



In [ ]:
delta_features = []

delta_features.append(
    add_daily_delta_features(
        drl_data,
        "pv_forecast_d_minus_1_1000_de_lu_mw",
        prefix="pv"
    )
)

delta_features.append(
    add_daily_delta_features(
        drl_data,
        "wind_onshore_forecast_d_minus_1_1000_de_lu_mw",
        prefix="wind_onshore"
    )
)

delta_features.append(
    add_daily_delta_features(
        drl_data,
        "wind_offshore_forecast_d_minus_1_1000_de_lu_mw",
        prefix="wind_offshore"
    )
)

delta_features.append(
    add_daily_delta_features(
        drl_data,
        "residual_load",
        prefix="residual_load"
    )
)


In [ ]:
daily_delta_all = pd.concat(delta_features, axis=1)

daily_agg = daily_agg.join(daily_delta_all)


In [ ]:

df_daily = daily_agg.join(ri_profit[["profit", "cycles"]], how="inner")

print(df_daily.shape)

df_daily

(1826, 65)


,epex_spot_60min_de_lu_eur_per_mwh_mean,epex_spot_60min_de_lu_eur_per_mwh_std,epex_spot_60min_de_lu_eur_per_mwh_min,epex_spot_60min_de_lu_eur_per_mwh_max,epex_spot_60min_de_lu_eur_per_mwh_q25,epex_spot_60min_de_lu_eur_per_mwh_q50,epex_spot_60min_de_lu_eur_per_mwh_q75,load_forecast_d_minus_1_1000_total_de_lu_mw_mean,load_forecast_d_minus_1_1000_total_de_lu_mw_std,load_forecast_d_minus_1_1000_total_de_lu_mw_min,...,pv_delta_sum,pv_delta_max,wind_onshore_delta_sum,wind_onshore_delta_max,wind_offshore_delta_sum,wind_offshore_delta_max,residual_load_delta_sum,residual_load_delta_max,profit,cycles
date,,,,,,,,,,,,,,,,,,,,,
2019-01-01 00:00:00+00:00,-6.875833,10.786639,-33.57,10.07,-10.5700,-4.930,0.0175,47744.070208,5133.243773,40308.2500,...,4424.875,816.7325,18310.9550,1729.1225,1606.9900,305.5500,35396.5875,2886.1950,68.233139,1.0
2019-01-02 00:00:00+00:00,29.104167,40.083914,-48.93,62.11,28.0700,47.815,55.2575,55944.564583,8385.547397,40825.7450,...,12974.745,2434.9175,22856.7100,2629.0125,2609.7275,435.5225,54600.7200,7504.9525,159.440419,2.0
2019-01-03 00:00:00+00:00,58.042083,9.148248,43.88,69.55,50.5425,60.875,65.7500,57691.780937,7265.181910,46078.2500,...,9828.700,1981.1900,7867.1500,855.9500,2637.3275,343.0850,42801.0300,5740.5400,56.598649,3.0
2019-01-04 00:00:00+00:00,48.917083,7.091700,26.90,55.78,46.9325,51.505,53.7100,55457.005625,6982.070073,43833.4025,...,3848.575,761.7675,16783.4850,1640.1500,3283.8425,634.4625,42034.4275,4489.0700,64.915423,4.0
2019-01-05 00:00:00+00:00,43.155417,14.022366,18.37,61.64,27.8500,50.125,52.1275,52843.881458,5805.721892,45028.6725,...,2741.525,504.0975,21259.6300,1494.2750,2362.4750,598.5150,42542.5050,6471.1975,102.020127,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-27 00:00:00+00:00,59.074583,24.869555,-0.02,82.73,51.9650,68.470,75.7525,53093.031354,7597.601289,39422.9500,...,19529.535,3543.5275,38360.0575,3935.0325,7738.5300,1334.7100,72683.9650,6577.7275,106.523683,88.0
2023-12-28 00:00:00+00:00,16.670833,16.803733,-1.43,46.83,0.0150,14.165,31.5025,55965.022708,7561.289589,43187.7150,...,15637.880,2820.3225,8205.3500,1143.6550,3266.9675,1130.9175,48287.1025,4767.0525,102.341338,89.0
2023-12-29 00:00:00+00:00,6.605000,8.989161,-0.95,26.59,-0.0250,1.660,10.0400,54341.245000,6885.347145,42627.0675,...,14551.115,2645.3100,11782.7250,1871.0975,2912.4800,1356.5150,43090.8850,4559.2200,91.073262,90.0


In [ ]:
# --- 4) Features + Target zusammenführen ---

# Annahme: ri_daily hat Spalten 'profit' und 'cycles'
ri_profit.index.name = "date"
df_daily = daily_agg.merge(
    ri_profit[["profit", "cycles"]],
    left_index=True,
    right_index=True,
    how="inner"
)

In [ ]:
#test_columns = ["profit","id_full_h_mean", "id_full_h_spread", "id_full_h_min", "id_full_h_max" , "id_full_h_std","id_full_h_q25",  "id_full_h_q50", "id_full_h_q75", "id_full_h_delta_sum", "id_full_h_delta_max"]
#df_daily = df_daily[test_columns].copy()
df_daily

,epex_spot_60min_de_lu_eur_per_mwh_mean,epex_spot_60min_de_lu_eur_per_mwh_std,epex_spot_60min_de_lu_eur_per_mwh_min,epex_spot_60min_de_lu_eur_per_mwh_max,epex_spot_60min_de_lu_eur_per_mwh_q25,epex_spot_60min_de_lu_eur_per_mwh_q50,epex_spot_60min_de_lu_eur_per_mwh_q75,load_forecast_d_minus_1_1000_total_de_lu_mw_mean,load_forecast_d_minus_1_1000_total_de_lu_mw_std,load_forecast_d_minus_1_1000_total_de_lu_mw_min,...,pv_delta_sum,pv_delta_max,wind_onshore_delta_sum,wind_onshore_delta_max,wind_offshore_delta_sum,wind_offshore_delta_max,residual_load_delta_sum,residual_load_delta_max,profit,cycles
date,,,,,,,,,,,,,,,,,,,,,
2019-01-01 00:00:00+00:00,-6.875833,10.786639,-33.57,10.07,-10.5700,-4.930,0.0175,47744.070208,5133.243773,40308.2500,...,4424.875,816.7325,18310.9550,1729.1225,1606.9900,305.5500,35396.5875,2886.1950,68.233139,1.0
2019-01-02 00:00:00+00:00,29.104167,40.083914,-48.93,62.11,28.0700,47.815,55.2575,55944.564583,8385.547397,40825.7450,...,12974.745,2434.9175,22856.7100,2629.0125,2609.7275,435.5225,54600.7200,7504.9525,159.440419,2.0
2019-01-03 00:00:00+00:00,58.042083,9.148248,43.88,69.55,50.5425,60.875,65.7500,57691.780937,7265.181910,46078.2500,...,9828.700,1981.1900,7867.1500,855.9500,2637.3275,343.0850,42801.0300,5740.5400,56.598649,3.0
2019-01-04 00:00:00+00:00,48.917083,7.091700,26.90,55.78,46.9325,51.505,53.7100,55457.005625,6982.070073,43833.4025,...,3848.575,761.7675,16783.4850,1640.1500,3283.8425,634.4625,42034.4275,4489.0700,64.915423,4.0
2019-01-05 00:00:00+00:00,43.155417,14.022366,18.37,61.64,27.8500,50.125,52.1275,52843.881458,5805.721892,45028.6725,...,2741.525,504.0975,21259.6300,1494.2750,2362.4750,598.5150,42542.5050,6471.1975,102.020127,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-27 00:00:00+00:00,59.074583,24.869555,-0.02,82.73,51.9650,68.470,75.7525,53093.031354,7597.601289,39422.9500,...,19529.535,3543.5275,38360.0575,3935.0325,7738.5300,1334.7100,72683.9650,6577.7275,106.523683,88.0
2023-12-28 00:00:00+00:00,16.670833,16.803733,-1.43,46.83,0.0150,14.165,31.5025,55965.022708,7561.289589,43187.7150,...,15637.880,2820.3225,8205.3500,1143.6550,3266.9675,1130.9175,48287.1025,4767.0525,102.341338,89.0
2023-12-29 00:00:00+00:00,6.605000,8.989161,-0.95,26.59,-0.0250,1.660,10.0400,54341.245000,6885.347145,42627.0675,...,14551.115,2645.3100,11782.7250,1871.0975,2912.4800,1356.5150,43090.8850,4559.2200,91.073262,90.0


In [ ]:

# --- 5) Random Forest Test ---

feature_cols = [
    col for col in df_daily.columns
    if col not in ["profit", "cycles"]
]

X = df_daily[feature_cols].values
y = df_daily["profit"].values

n = len(df_daily)
split = int(n * 0.8)

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Val   R2:", r2_score(y_val, y_val_pred))
print("Val  MAE:", mean_absolute_error(y_val, y_val_pred))


Train R2: 0.9831271542471811
Val   R2: 0.44703080104329596
Val  MAE: 54.76917919444081


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

# --- 1) Features definieren ---
feature_cols = [
    col for col in df_daily.columns
    if col not in ["profit", "cycles"]
]

X = df_daily[feature_cols].values
y = df_daily["profit"].values

# --- 2) NaN-Fix: Imputer ---
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

# --- 3) Train/Test-Split ---
n = len(df_daily)
split = int(n * 0.8)

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

# --- 4) Gradient Boosting Modell ---
model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

# --- 5) Vorhersagen ---
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

# --- 6) Metriken ---
print("Train R2:", r2_score(y_train, y_train_pred))
print("Val   R2:", r2_score(y_val, y_val_pred))
print("Val  MAE:", mean_absolute_error(y_val, y_val_pred))



Train R2: 0.9685372460821843
Val   R2: 0.4544649409310981
Val  MAE: 54.54446632214538


In [ ]:
fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
})

fi = fi.sort_values("importance", ascending=False)

print("\nFeature Importances (sortiert):")
print(fi)



Feature Importances (sortiert):
                                    feature  importance
1     epex_spot_60min_de_lu_eur_per_mwh_std    0.326338
49                              epex_spread    0.286956
43                            id_full_h_std    0.188657
45                            id_full_h_max    0.054923
3     epex_spot_60min_de_lu_eur_per_mwh_max    0.037014
..                                      ...         ...
52                              day_of_week    0.000111
55                             pv_delta_sum    0.000111
17  pv_forecast_d_minus_1_1000_de_lu_mw_max    0.000046
18  pv_forecast_d_minus_1_1000_de_lu_mw_q25    0.000015
16  pv_forecast_d_minus_1_1000_de_lu_mw_min    0.000000

[63 rows x 2 columns]


In [ ]:
# Pearson-Korrelation jeder Variable mit profit
corr_with_profit = (
    df_daily
    .corr(method="pearson")["profit"]
    .sort_values(ascending=False)
)

corr_with_profit.to_csv("feature_correlation_with_profit.csv")